In [2]:
import re
import pandas as pd
import os 

notebook_path= os.getcwd()
in_dir_notifications  = os.path.abspath(os.path.join(notebook_path,"..","..","Data","notification_historical","updated_notifications.jsonl"))
in_dir_master_direction = os.path.abspath(os.path.join(notebook_path,"..","..","Data","master_directory","master_directory.jsonl"))
circulars = pd.read_json(in_dir_notifications, lines=True)
master_directions = pd.read_json(in_dir_master_direction, lines = True)
# regexes to catch "Reserve Bank of India (X) ... Directions" and the no-paren variant
p1 = re.compile(r"Reserve Bank of India\s*[-–—]?\s*\(([^)]+)\)\s*(?:[A-Za-z]+\s+){0,3}Directions", re.IGNORECASE)
p2 = re.compile(r"Reserve Bank of India\s*[-–—]\s*([A-Za-z ,]+?),?\s*Directions", re.IGNORECASE)

norm = lambda s: re.sub(r"\s+", " ", s.replace("–","-").replace("—","-")).strip(" ,-").lower()

# extract candidate names from title + text
circulars["found_title"] = circulars["title"].apply(lambda t: [norm(x) for x in (p1.findall(t) + p2.findall(t))] if isinstance(t, str) else [])
circulars["found_text"]  = circulars["text"].apply(lambda t: [norm(x) for x in (p1.findall(t) + p2.findall(t))] if isinstance(t, str) else [])

# build master lookup: normalized name -> master direction id
master_lookup = {}
for _, r in master_directions.iterrows():
    for name in [norm(x) for x in (p1.findall(r["title"]) + p2.findall(r["title"]))]:
        master_lookup[name] = r["id"]

# match: prefer title hit, fall back to text hit
def _match(row):
    for n in row["found_title"]:
        if n in master_lookup: return master_lookup[n], "title_match"
    for n in row["found_text"]:
        if n in master_lookup: return master_lookup[n], "text_match"
    return None, "no_match",

circulars[["matched_id", "match_method"]] = circulars.apply(lambda r: pd.Series(_match(r)), axis=1)

# stats
print(circulars["match_method"].value_counts())
print(f"matched: {(circulars['match_method']!='no_match').mean():.1%}")

circulars[["id","title","found_title","matched_id","match_method"]]

match_method
no_match       97
title_match    34
text_match      7
Name: count, dtype: int64
matched: 29.7%


,id,title,found_title,matched_id,match_method
0,13167,Reserve Bank of India (Setting Up of Wholly Ow...,[],NaN,no_match
1,13168,Reserve Bank of India (Universal Banks – Licen...,[],NaN,no_match
2,13169,Compliance with Know Your Customer (KYC) norms,[],12943.0,text_match
3,13170,Consolidation of Regulations – Withdrawal of c...,[],NaN,no_match
4,13171,Compliance with Know Your Customer (KYC) norms,[],NaN,no_match
...,...,...,...,...,...
133,13675,Formation of new districts in the Union Territ...,[],NaN,no_match
134,13676,"Implementation of Section 51A of UAPA, 1967: U...",[],NaN,no_match
135,13677,"Implementation of Section 51A of UAPA, 1967: U...",[],NaN,no_match
136,13678,"Implementation of Section 51A of UAPA, 1967: U...",[],NaN,no_match


In [4]:
no_match = circulars[circulars["match_method"] == "no_match"]
print(len(no_match))
for t in no_match["title"].head(10):
    print("-", t)


97
- Reserve Bank of India (Setting Up of Wholly Owned Subsidiaries by Foreign Banks) Guidelines, 2025 (Updated as on April 1, 2026)
- Reserve Bank of India (Universal Banks – Licensing) Guidelines, 2025
- Consolidation of Regulations – Withdrawal of circulars
- Compliance with Know Your Customer (KYC) norms
- Liberalised Remittance Scheme (LRS)- Submission of ‘LRS Daily Return’ by Authorised Dealers- Category -II banks/ entities and Full- Fledged Money Changers
- Liquidity Adjustment Facility - Change in rates
- Standing Liquidity Facility for Primary Dealers
- Penal Interest on shortfall in CRR and SLR requirements - Change in Bank Rate
- Reserve Bank of India (Non-Operative Financial Holding Company) (Amendment) Directions, 2025
- Export and Import of Indian Currency to or from Nepal and Bhutan


In [8]:
ref_pattern = re.compile(r"([A-Z]+(?:\.[A-Z]+)+\.\d+)/([\d-]+)/(\d{4}-\d{2})")

circulars["subject_code"] = circulars["text"].str.extract(ref_pattern)[1]
master_directions["subject_code"] = master_directions["text"].str.extract(ref_pattern)[1]

In [9]:
dupe_codes = master_directions.dropna(subset=["subject_code"]).groupby("subject_code")["id"].nunique()
dupe_codes[dupe_codes > 1]

subject_code
33-01-010    30
Name: id, dtype: int64

In [10]:
code_matches = circulars.merge(
    master_directions.dropna(subset=["subject_code"])[["id","subject_code"]],
    on="subject_code", how="left", suffixes=("", "_master")
)

# where code-match succeeds but your text/title match didn't
recovered = code_matches[(code_matches["match_method"] == "no_match") & code_matches["id_master"].notna()]
print(f"{len(recovered)} previously unmatched circulars recoverable via subject_code")

# where both methods matched, do they point to the same master direction?
both = code_matches[(code_matches["match_method"] != "no_match") & code_matches["id_master"].notna()]
disagreements = both[both["matched_id"] != both["id_master"]]
print(f"{len(disagreements)} / {len(both)} disagree between text-match and code-match")

30 previously unmatched circulars recoverable via subject_code
0 / 4 disagree between text-match and code-match


In [18]:
pd.set_option('display.max_colwidth',None)
master_directions[master_directions["subject_code"] == "33-01-010"][["id", "title"]]

,id,title
0,12931,"Reserve Bank of India (Non-Banking Financial Companies – Miscellaneous) Directions, 2025 (Updated as on February 26, 2026)"
1,12932,"Reserve Bank of India (Non-Operative Financial Holding Companies) Directions, 2025 (Updated as on December 05, 2025)"
2,12933,"Reserve Bank of India (Non-Banking Financial Companies – Microfinance Institution) Directions, 2025"
3,12934,"Reserve Bank of India (Non-Banking Financial Companies – Peer to Peer Lending Platform) Directions, 2025"
4,12935,"Reserve Bank of India (Mortgage Guarantee Companies) Directions, 2025 (Updated as on March 10, 2026)"
5,12936,"Reserve Bank of India (Non-Banking Financial Companies - Account Aggregator) Directions, 2025"
6,12937,"Reserve Bank of India (Core Investment Companies) Directions, 2025 (Updated as on March 10, 2026)"
7,12938,"Reserve Bank of India (Standalone Primary Dealers) Directions, 2025 (Updated as on March 10, 2026)"
9,12940,"Reserve Bank of India (Non-Banking Financial Companies – Climate Finance and Management of Climate Change Risks) Directions, 2025"
10,12941,"Reserve Bank of India (Non-Banking Financial Companies – Managing Risks in Outsourcing) Directions, 2025"


In [19]:
recovered["id"].value_counts()

id
13170    30
Name: count, dtype: int64

In [20]:
clean_master = master_directions[master_directions["subject_code"] != "33-01-010"]

code_matches = circulars.merge(
    clean_master.dropna(subset=["subject_code"])[["id", "subject_code"]],
    on="subject_code", how="left", suffixes=("", "_master")
)

recovered = code_matches[(code_matches["match_method"] == "no_match") & code_matches["id_master"].notna()]
print(recovered["id"].value_counts())  # sanity check: every id should now appear exactly once

Series([], Name: count, dtype: int64)


In [21]:
codes_in_master = set(clean_master["subject_code"].dropna())
circulars_with_relevant_code = circulars[circulars["subject_code"].isin(codes_in_master)]
print(len(circulars_with_relevant_code))

4
